# Orchestrator Colab Worker

This notebook keeps only the minimum bootstrap logic inline: mount Drive, configure runtime env, clone or refresh the repository, and invoke the checked-out Colab worker CLI with Colab-visible logs. The notebook is still usable from a single-file starting point.


## Drive


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## Notebook bootstrap


In [ ]:
import json
import os
import shlex
import subprocess
import sys
import time
from pathlib import Path

REPO_URL = "https://github.com/discoverex/orchestrator.git"
REPO_BRANCH = "dev"
DRIVE_ROOT = Path("/content/drive/MyDrive/discoverex")
REPO_DIR = DRIVE_ROOT / "orchestrator"
CACHE_ROOT = DRIVE_ROOT / "cache"
CHECKPOINT_DIR = Path("/content/drive/MyDrive/orchestrator/checkpoints")
VENV_DIR = Path("/content/venv")
PID_FILE = CACHE_ROOT / "colab-worker.pid"
LOG_FILE = CACHE_ROOT / "colab-worker.log"
RUNNER_PATH = REPO_DIR / "infra" / "stacks" / "worker" / "colab" / "colab_worker_runner.py"


def format_command(cmd: list[str]) -> str:
    return " ".join(shlex.quote(part) for part in cmd)


def run_step(
    cmd: list[str],
    *,
    cwd: Path | None = None,
    env: dict[str, str] | None = None,
    check: bool = True,
    step: str,
) -> str:
    started = time.monotonic()
    print(f"[{step}] START {format_command(cmd)}", flush=True)
    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    lines: list[str] = []
    assert proc.stdout is not None
    for line in proc.stdout:
        lines.append(line)
        print(line, end="")
    rc = proc.wait()
    duration = time.monotonic() - started
    if rc != 0 and check:
        print(f"[{step}] FAILED rc={rc} after {duration:.1f}s", flush=True)
        raise RuntimeError(f"{step} failed with rc={rc}: {format_command(cmd)}")
    state = "DONE" if rc == 0 else "FAILED"
    print(f"[{step}] {state} rc={rc} after {duration:.1f}s", flush=True)
    return "".join(lines)


def clean_env_value(value: str | None) -> str:
    if value is None:
        return ""
    normalized = value.strip()
    if len(normalized) >= 2 and normalized[0] == normalized[-1] and normalized[0] in ("'", '"'):
        normalized = normalized[1:-1].strip()
    return normalized


def configure_env(secret_reader=None) -> dict[str, str]:
    defaults = {
        "PREFECT_API_URL": "https://prefect-api.discoverex.qzz.io/api",
        "PREFECT_WORK_POOL": "gpu-pool",
        "PREFECT_WORK_QUEUE": "gpu-colab",
        "ORCHESTRATOR_CHECKPOINT_DIR": str(CHECKPOINT_DIR),
        "PIP_CACHE_DIR": str(CACHE_ROOT / "pip"),
        "XDG_CACHE_HOME": str(CACHE_ROOT / "xdg"),
    }
    optional_keys = (
        "PREFECT_CF_ACCESS_CLIENT_ID",
        "PREFECT_CF_ACCESS_CLIENT_SECRET",
        "CF_ACCESS_CLIENT_ID",
        "CF_ACCESS_CLIENT_SECRET",
        "STORAGE_GATEWAY_TOKEN",
        "STORAGE_GATEWAY_URL",
    )
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    (CACHE_ROOT / "pip").mkdir(parents=True, exist_ok=True)
    (CACHE_ROOT / "xdg").mkdir(parents=True, exist_ok=True)
    os.environ["PYTHONPATH"] = str(REPO_DIR / "src")
    for key, default in defaults.items():
        current = clean_env_value(os.environ.get(key))
        if current:
            os.environ[key] = current
            continue
        secret_value = ""
        if secret_reader is not None:
            try:
                secret_value = clean_env_value(secret_reader(key))
            except Exception:
                secret_value = ""
        os.environ[key] = secret_value or default
    for key in optional_keys:
        current = clean_env_value(os.environ.get(key))
        if current:
            os.environ[key] = current
            continue
        secret_value = ""
        if secret_reader is not None:
            try:
                secret_value = clean_env_value(secret_reader(key))
            except Exception:
                secret_value = ""
        if secret_value:
            os.environ[key] = secret_value
    cf_id = os.environ.get("PREFECT_CF_ACCESS_CLIENT_ID") or os.environ.get("CF_ACCESS_CLIENT_ID")
    cf_secret = os.environ.get("PREFECT_CF_ACCESS_CLIENT_SECRET") or os.environ.get("CF_ACCESS_CLIENT_SECRET")
    if cf_id and cf_secret:
        os.environ["PREFECT_CLIENT_CUSTOM_HEADERS"] = json.dumps(
            {
                "CF-Access-Client-Id": cf_id,
                "CF-Access-Client-Secret": cf_secret,
            },
            ensure_ascii=True,
        )
    else:
        os.environ.pop("PREFECT_CLIENT_CUSTOM_HEADERS", None)
    snapshot = {
        "repo_dir": str(REPO_DIR),
        "cache_root": str(CACHE_ROOT),
        "venv_dir": str(VENV_DIR),
        "checkpoint_dir": str(CHECKPOINT_DIR),
        "pid_file": str(PID_FILE),
        "log_file": str(LOG_FILE),
        "prefect_api_url": os.environ.get("PREFECT_API_URL", ""),
        "prefect_work_pool": os.environ.get("PREFECT_WORK_POOL", ""),
        "prefect_work_queue": os.environ.get("PREFECT_WORK_QUEUE", ""),
        "has_prefect_custom_headers": bool(os.environ.get("PREFECT_CLIENT_CUSTOM_HEADERS")),
        "has_storage_gateway_url": bool(os.environ.get("STORAGE_GATEWAY_URL")),
        "has_storage_gateway_token": bool(os.environ.get("STORAGE_GATEWAY_TOKEN")),
    }
    print(json.dumps(snapshot, indent=2, sort_keys=True))
    return os.environ.copy()


def ensure_repo_checkout() -> None:
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    if not REPO_DIR.exists():
        run_step(["git", "clone", REPO_URL, str(REPO_DIR)], cwd=DRIVE_ROOT, step="repo")
    else:
        print(f"[repo] using existing repo at {REPO_DIR}", flush=True)
    run_step(["git", "fetch", "--all", "--prune"], cwd=REPO_DIR, step="repo")
    run_step(["git", "checkout", REPO_BRANCH], cwd=REPO_DIR, step="repo")
    run_step(["git", "fetch", "origin"], cwd=REPO_DIR, step="repo")
    run_step(["git", "reset", "--hard", f"origin/{REPO_BRANCH}"], cwd=REPO_DIR, step="repo")


def runner_command(command: str, *extra: str) -> list[str]:
    base = [
        str(VENV_DIR / "bin" / "python") if command != "bootstrap" else sys.executable,
        str(RUNNER_PATH),
        "--repo-dir",
        str(REPO_DIR),
        "--cache-root",
        str(CACHE_ROOT),
        "--venv-dir",
        str(VENV_DIR),
        "--checkpoint-dir",
        str(CHECKPOINT_DIR),
        "--pid-file",
        str(PID_FILE),
        "--log-file",
        str(LOG_FILE),
    ]
    if command == "logs":
        base.extend(["--tail", "120"])
    base.extend(extra)
    base.append(command)
    return base


def run_runner(command: str, *extra: str, env: dict[str, str] | None = None) -> str:
    return run_step(runner_command(command, *extra), cwd=REPO_DIR, env=env, step=f"runner:{command}")


## Runtime config


In [ ]:
try:
    from google.colab import userdata
except ImportError:
    userdata = None


def secret_reader(key: str) -> str | None:
    if userdata is None:
        return None
    try:
        return userdata.get(key)
    except Exception:
        return None


RUNTIME_ENV = configure_env(secret_reader=secret_reader)


## Repo sync


In [ ]:
ensure_repo_checkout()


## Bootstrap runtime


In [ ]:
run_runner("bootstrap", env=RUNTIME_ENV)


## Start worker


In [ ]:
run_runner("stop", env=RUNTIME_ENV)
run_runner("start", "--skip-install", env=RUNTIME_ENV)
run_runner("status", env=RUNTIME_ENV)
run_runner("logs", env=RUNTIME_ENV)


## Status and logs


In [ ]:
run_runner("status", env=RUNTIME_ENV)
run_runner("logs", env=RUNTIME_ENV)


## Stop worker


In [ ]:
run_runner("stop", env=RUNTIME_ENV)
